# Part 1. Equation of a Slime

How many late days are you using for this assignment? 0

In [1]:
import pandas as pd


## 1. Loading the dataset

In [70]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
scienceData = pd.read_csv('science_data_large.csv')
display(scienceData.head(15))
print(scienceData.info())


,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 23.6 KB
None


## 2. Splitting the dataset

In [116]:
# Take the pandas dataset and split it into our features (X) and label (y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 
from sklearn.model_selection import train_test_split

X = scienceData.drop('Size nm^3', axis=1)
y = scienceData['Size nm^3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, shuffle=True)

## 3. Perform a Linear Regression

In [117]:
# Use sklearn to train a model on the training set

# Create a sample datapoint and predict the output of that sample with the trained model

# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

sample = X.iloc[0]
sample_df = pd.DataFrame([sample], columns=X.columns)
print(model.predict(sample_df).round(5))

print(model.score(X_test, y_test))
print(model.coef_.round(5))
print(model.intercept_.round(5))

[664984.8963]
0.8552472077276095
[ 866.14641 1032.69507]
-409391.47958


Write the linear equation of a slime: (example equation: $E = mc^2$)<br>
$S = 866.14641x_1 + 1032.69507x_2 - 409391.47958$<br>
x: temperature(degrees C)<br>
y: Mols of KCL

Report on score and explain meaning: The score is approximately 0.8552 which represents the R^2 value. The R^2 value is a measure of well the model fits the data, more specifically how much of the variance in the dependent variable can be explained by the independent variable.

## 4. Use Cross Validation

In [118]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42

# Report on their finding and their significance
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
print(scores)
print(scores.mean())
print(scores.std())


[0.83918826 0.87051239 0.85871066 0.87202623 0.84364641]
0.8568167899144437
0.013466307372096016


Write findings here: The five scores for each fold returned the scores  0.83918826, 0.87051239, 0.85871066, 0.87202623, 0.84364641 which has a mean of approximately 0.8568 and the standard deviation 0.0135. The low standard deviation tells us that the performance is pretty consistent across the different folds, meaning it is likely better at generalizing to new data. 


## 5. Using Polynomial Regression

In [119]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)

from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
#[1, x1, x2, x1^2, x1*x2, x2^2]

newModel = LinearRegression()
newModel = newModel.fit(X_poly, y)

scores = cross_val_score(newModel, X_poly, y, cv=5)
print(newModel.coef_.round(5))
print(newModel.intercept_.round(5))
print(scores)
print(scores.mean())
print(scores.std())
# Report on the metrics and output the resultant equation as you did in Part 3.

[ 0.      12.      -0.      -0.       2.       0.02857]
2e-05
[1. 1. 1. 1. 1.]
1.0
0.0


Write the polynomial equation of a slime: (example equation: $E = mc^2$)<br>
$S = 12x_1 + 2x_1x_2 + 0.02857x_2^2 - 0.00002$

Report on the score and interpret: The score is 1 because the new model perfectly fits the dataset. This means the predictions made by the model are the exact values. This means the size of a slime is directly related to temperature and amount of KCL.

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [193]:
# Load the dataset. Then train and evaluate the classification models.
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

kidneyData = pd.read_csv('ckd_feature_subset.csv')
X = kidneyData.drop('Target_ckd', axis=1)
y = kidneyData['Target_ckd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, shuffle=True)

models = {
    'logReg': LogisticRegression(random_state=42),
    'SVM': SVC(kernel='linear', random_state=42),
    'kNN': KNeighborsClassifier(n_neighbors=5),
    'NeuralNet': MLPClassifier(random_state=42, max_iter=1000)
}
trainedModels = {}
results = {}

for name, model in models.items():
    trainedModels[name] = model.fit(X_train, y_train)
    scores = cross_val_score(trainedModels[name], X, y, cv=5)
    results[name] = {'Mean Accuracy': scores.mean().round(5), 'Std Dev': scores.std().round(5)}

resultsDF = pd.DataFrame(results)
print(resultsDF)

                logReg      SVM      kNN  NeuralNet
Mean Accuracy  0.85613  0.86301  0.92817    0.94129
Std Dev        0.05424  0.05650  0.03189    0.02404


## Results and Conclusion for Classification Experiments

The Neural network model has the highest mean accuracy of 0.9413 with a standard deviation of 0.024 while the logistic Regression model had the lowest Mean Accuracy at approximately 0.85613. I think the neural network performed best because it's able to handle potentially non-linear relationships with CKD risk. It is possible that normalization of the data makes the nearest-neighbor model worse since it more closely aligns with specific features. 

## Results and Conclusion for Neural Network Experiments

In [203]:
# Experiments with Neural Network.
nn_configs = {
    'nn1': MLPClassifier(hidden_layer_sizes=(100, 50,), random_state=42, max_iter=1000),
    'nn2': MLPClassifier(hidden_layer_sizes=(100, 100, 50,), random_state=42, max_iter=1000),
    'nn3': MLPClassifier(hidden_layer_sizes=(100, 50,),activation='tanh', random_state=42, max_iter=1000),
    'nn4': MLPClassifier(hidden_layer_sizes=(100,), activation='tanh', random_state=42, max_iter=1000),
    'nn5': MLPClassifier(hidden_layer_sizes=(100,), solver='lbfgs', random_state=42, max_iter=500),
}
nn_results = {}
scores = cross_val_score(models['NeuralNet'], X, y, cv=5)

nn_results['nnBase'] = {'Mean Accuracy': scores.mean().round(5), 'Std Dev': scores.std().round(5)}

for name, model in nn_configs.items():
    scores = cross_val_score(model, X, y, cv=5)
    nn_results[name] = {'Mean Accuracy': scores.mean().round(5), 'Std Dev': scores.std().round(5)}
nn_results_df = pd.DataFrame(nn_results).T
print(nn_results_df)

        Mean Accuracy  Std Dev
nnBase        0.94129  0.02404
nn1           0.97376  0.02483
nn2           0.97398  0.02420
nn3           0.96065  0.02455
nn4           0.93462  0.02936
nn5           0.96731  0.02934


Comparing these results to the base Neural Network, we can see that the hidden layer sizes and number of layers have the most impact on the result while solver helps but can slightly increases the standard deviation. Activation hurts the accuracy and standard deviation. 